# PRISM Training — Colab

**Single file**: trains PRISM end-to-end (Stage 1 distill + Stage 2 RL).

Switch `DATA_MODE` in Cell 4 between:
- `'synthetic'` — toy KGQA, free Colab OK, ~30 min total
- `'rog'` — real WebQSP via HuggingFace `rmanluo/RoG-webqsp`, ~2-3 h on T4

Output checkpoints + metrics auto-saved to Drive.

## Cell 1 — Mount Drive + install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torch numpy scikit-learn tqdm matplotlib datasets sentence-transformers

## Cell 2 — Setup paths

Adjust `REPO_ROOT` to point at the **parent of `colab/`** in your Drive.

In [ ]:
import os, sys, pathlib

REPO_ROOT = '/content/drive/MyDrive/ThS/Research/colab'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Ensure namespace packages exist (case-sensitive — match folder name on Drive)
for sub in ['COLAB', 'COLAB/graph_rag']:
    pathlib.Path(f'{REPO_ROOT}/{sub}/__init__.py').touch()

# Verify
from COLAB.graph_rag.prism.config import CFG
print('Path OK. Initial CFG:', CFG)

## Cell 3 — Imports

In [ ]:
import json, datetime, torch
import matplotlib.pyplot as plt

from COLAB.graph_rag.prism.data import SyntheticKGQA, RoGDataset
from COLAB.graph_rag.prism.las import LAS
from COLAB.graph_rag.prism.controller import deterministic_select
from COLAB.graph_rag.prism.distill import train_distill
from COLAB.graph_rag.prism.rl import train_rl
from COLAB.graph_rag.prism.evaluate import evaluate, baseline_full_subgraph
from COLAB.graph_rag.prism.metrics import (
    expected_calibration_error, brier_score, hit_at_k, reliability_diagram
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0),
          f'({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')
else:
    print('No GPU — synthetic mode only.')

## Cell 4 — Pick data mode

**`'synthetic'`** — start here first to verify pipeline (~30 min).
**`'rog'`** — only after synthetic works.

In [ ]:
DATA_MODE = 'synthetic'   # 'synthetic' or 'rog'

OUTPUT_DIR = f'{REPO_ROOT}/prism_outputs/{DATA_MODE}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CFG.out_dir = OUTPUT_DIR
print('Outputs ->', OUTPUT_DIR)

## Cell 5 — Config

In [ ]:
# Architecture
CFG.d_model       = 256
CFG.n_layers      = 4
CFG.max_paths     = 50
CFG.max_path_len  = 3
CFG.n_candidates  = 32
CFG.d_query       = 384

# Controller
CFG.stopping_eps  = 0.05
CFG.diversity_lambda = 0.3
CFG.max_select_paths = 10

# Training
if DATA_MODE == 'synthetic':
    CFG.n_train = 2000
    CFG.n_eval = 400
    CFG.n_relations = 64
    CFG.distill_epochs = 5
    CFG.distill_batch_size = 8
    CFG.rl_steps = 500
    CFG.rl_n_samples = 4
else:  # rog
    MAX_TRAIN, MAX_EVAL = 1500, 300
    CFG.distill_epochs = 5
    CFG.distill_batch_size = 4
    CFG.rl_steps = 1000
    CFG.rl_n_samples = 3

print(CFG)

## Cell 6 — Load data

Branches on `DATA_MODE`.

In [ ]:
if DATA_MODE == 'synthetic':
    train_set = SyntheticKGQA(CFG.n_train, cfg=CFG, seed=CFG.seed)
    eval_set  = SyntheticKGQA(CFG.n_eval,  cfg=CFG, seed=CFG.seed + 100)
    rel2id, ent2id = None, None

elif DATA_MODE == 'rog':
    CACHE_DIR = f'{REPO_ROOT}/rog_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    train_set = RoGDataset.from_hf(
        'rmanluo/RoG-webqsp', split='train',
        sbert_model='all-MiniLM-L6-v2',
        cache_dir=CACHE_DIR, cfg=CFG, max_items=MAX_TRAIN,
    )
    eval_set = RoGDataset.from_hf(
        'rmanluo/RoG-webqsp', split='test',
        sbert_model='all-MiniLM-L6-v2',
        cache_dir=CACHE_DIR, cfg=CFG, max_items=MAX_EVAL,
    )
    rel2id, ent2id = train_set.rel2id, train_set.ent2id
    # CRITICAL: bump CFG vocab to match real data BEFORE LAS init
    CFG.n_relations = train_set.n_relations
    CFG.n_entities  = train_set.n_entities
    print(f'CFG.n_relations -> {CFG.n_relations}, CFG.n_entities -> {CFG.n_entities}')

else:
    raise ValueError(f'Unknown DATA_MODE={DATA_MODE}')

print(f'train={len(train_set)}  eval={len(eval_set)}')

## Cell 7 — Inspect one sample (sanity check)

In [ ]:
s = train_set[0]
print(f'qid={s.qid}')
print(f'query_emb {s.query_emb.shape}, paths {s.paths.shape}, '
      f'candidates {s.candidate_ids.shape[0]}, gold {s.gold_entities.tolist()}')
print(f'teacher_dist top-5: {s.teacher_distribution.topk(5).values.tolist()}')

if DATA_MODE == 'rog' and rel2id is not None:
    id2rel = {v:k for k,v in rel2id.items()}
    id2ent = {v:k for k,v in ent2id.items()}
    print('\nFirst 3 paths:')
    for i in range(min(3, s.paths.size(0))):
        rels = [id2rel.get(int(r),'?') for r in s.paths[i] if r != 0]
        cand_idx = int(s.path_to_candidate[i])
        cand_name = id2ent.get(int(s.candidate_ids[cand_idx]), '?')
        print(f'  {" → ".join(rels)} → {cand_name}')

## Cell 8 — Instantiate LAS

In [ ]:
las = LAS(CFG).to(device)
n_params = sum(p.numel() for p in las.parameters() if p.requires_grad)
print(f'LAS trainable params: {n_params:,}')

# Verify forward works on real data
with torch.no_grad():
    s = train_set[0]
    for k_ in ['query_emb','paths','path_to_candidate','candidate_gnn_scores',
               'candidate_ids','gold_entities']:
        setattr(s, k_, getattr(s, k_).to(device))
    probs, conf, _ = las(s.query_emb, s.paths, s.path_to_candidate, s.candidate_gnn_scores)
    print(f'forward OK: probs {probs.shape}, sum={float(probs.sum()):.4f}, conf={float(conf):.3f}')

## Cell 9 — Stage 1: Distillation

Synthetic: ~5-15 min on T4. RoG: ~30-60 min on T4.

In [ ]:
ckpt_distill = train_distill(las, train_set, eval_set, cfg=CFG)
print('Saved checkpoint:', ckpt_distill)

## Cell 10 — Eval after Stage 1

In [ ]:
m_stage1 = evaluate(las, eval_set, CFG)
print('After Stage 1:')
for k, v in m_stage1.items():
    print(f'  {k:14s} = {v}')

## Cell 11 — Stage 2: REINFORCE

Synthetic: ~5-10 min. RoG: ~30-90 min.

In [ ]:
ckpt_rl = train_rl(las, train_set, eval_set, cfg=CFG)
print('Saved checkpoint:', ckpt_rl)

## Cell 12 — Final evaluation

In [ ]:
m_prism = evaluate(las, eval_set, CFG)
m_base  = baseline_full_subgraph(eval_set)

print('=' * 60)
print('PRISM:')
for k, v in m_prism.items(): print(f'  {k:14s} = {v}')
print('\nBaseline (full subgraph):')
for k, v in m_base.items(): print(f'  {k:14s} = {v}')
print('\nΔ Hit@1     = {:+.4f}'.format(m_prism['hit@1'] - m_base['hit@1']))
print('Cost saving  = {:.1f}% paths fewer'.format(
    100 * (1 - m_prism['mean_paths'] / max(1e-6, m_base['mean_paths']))))

## Cell 13 — Calibration analysis

In [ ]:
confs, corrs = [], []
with torch.no_grad():
    iterable = eval_set if DATA_MODE == 'rog' else eval_set.samples
    for sample in iterable:
        for k_ in ['query_emb','paths','path_to_candidate','candidate_gnn_scores',
                   'candidate_ids','gold_entities']:
            setattr(sample, k_, getattr(sample, k_).to(device))
        trace = deterministic_select(las, sample, CFG)
        confs.append(float(trace.confidence.item()))
        corrs.append(hit_at_k(trace.final_probs, sample.candidate_ids,
                              sample.gold_entities, k=1))

ece = expected_calibration_error(confs, corrs)
brier = brier_score(confs, corrs)
print(f'ECE   = {ece:.4f}')
print(f'Brier = {brier:.4f}')

centers, accs, _ = reliability_diagram(confs, corrs, n_bins=10)
plt.figure(figsize=(5,5))
plt.plot([0,1],[0,1],'k--',alpha=0.4,label='perfect')
plt.bar(centers, accs, width=0.08, alpha=0.6, edgecolor='black', label='PRISM')
plt.xlabel('confidence'); plt.ylabel('accuracy')
plt.title(f'Reliability — {DATA_MODE}'); plt.legend()
plt.xlim(0,1); plt.ylim(0,1); plt.show()

## Cell 14 — Pareto cost-quality sweep

In [ ]:
eps_grid = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
frontier = []
for eps in eps_grid:
    CFG.stopping_eps = eps
    m = evaluate(las, eval_set, CFG)
    frontier.append((m['mean_paths'], m['hit@1']))
    print(f'eps={eps:>6.3f}  paths={m["mean_paths"]:5.2f}  hit@1={m["hit@1"]:.3f}')

xs, ys = zip(*frontier)
plt.figure(figsize=(6,4))
plt.plot(xs, ys, 'o-', label='PRISM Pareto')
plt.scatter([m_base['mean_paths']],[m_base['hit@1']],
            color='red', s=80, label='Baseline (full S)', zorder=5)
plt.xlabel('Mean paths used'); plt.ylabel('Hit@1')
plt.title(f'PRISM Pareto — {DATA_MODE}'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

CFG.stopping_eps = 0.05   # reset

## Cell 15 — Save all results to Drive

In [ ]:
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
results = {
    'timestamp': ts,
    'data_mode': DATA_MODE,
    'config': {k:v for k,v in CFG.__dict__.items() if not k.startswith('_')},
    'stage1_metrics': m_stage1,
    'final_metrics': m_prism,
    'baseline_metrics': m_base,
    'calibration': {'ece': ece, 'brier': brier},
    'pareto_frontier': frontier,
}
out_path = f'{OUTPUT_DIR}/results_{ts}.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print('Results saved:', out_path)
print('Checkpoints saved at:')
print(' ', ckpt_distill)
print(' ', ckpt_rl)

## Next steps

1. **First time**: run with `DATA_MODE='synthetic'` end-to-end to verify pipeline.
2. **Phase 2**: switch to `DATA_MODE='rog'`, rerun cells 4 onwards.
3. **Phase 3** (later): subclass `TeacherLLM` to call real Llama for actual LLM grounding.